# Capstone — Refresh / Content Opportunity Scoring

This notebook supports the deployed research paper. It uses the FlyRank internship warehouse to rank content pages for refresh review using observed search and engagement signals.

The notebook is public-safe: it uses anonymized IDs and aggregated metrics only. No client names, domains, private queries, or credentials are included.

## 1. Question

**Research question:** Can observed search and content signals be used to rank pages that are worth reviewing for a content refresh?

**Decision supported:** Which pages should a content team review first when review time is limited?

The output is **directional decision-support**. It does not claim that refreshing a page causes better Google performance.

In [ ]:
print("Lane: Refresh / Content Opportunity Scoring")
print("Decision: Which pages should be reviewed first?")

## 2. Data

I use the FlyRank internship warehouse release, specifically the `fact_content_daily_performance` table.

**Prediction window:** 2026-03-01 through 2026-04-30.

**Future outcome window:** 2026-05-01 through 2026-06-30.

The unit of analysis is one anonymized client/content page pair. Features are aggregated only from the prediction window; the future window is used only to define the observed outcome.

Rows are kept when both windows have at least 100 GSC impressions. Pages with zero current CTR are excluded from modeling because percentage CTR change is undefined.

No client names, domains, URLs, private queries, or credentials are used in the analysis.

In [ ]:
import os
import getpass
import duckdb
import pandas as pd
import numpy as np

# The token is entered interactively and is never written into the notebook.
HF_TOKEN = getpass.getpass("Enter your Hugging Face READ token: ")

con = duckdb.connect()

con.execute(
    f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

warehouse = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/**/*.parquet"
)

print("Warehouse connection ready.")
print(con.sql(f"SELECT COUNT(*) AS rows FROM read_parquet('{warehouse}')").df())

In [ ]:
# Basic warehouse scope check
con.sql(f'''
SELECT
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date,
    COUNT(DISTINCT content_hash_id) AS pages,
    COUNT(DISTINCT client_hash_id) AS clients
FROM read_parquet('{warehouse}')
''').df()

## 3. Methodology

### Features
The model uses signals available before the future window:

- GSC impressions
- GSC clicks
- average position
- GA4 sessions
- GA4 engaged sessions
- scroll events
- number of observed days
- current CTR

### Label
A page is labeled `1` when its future CTR is at least **20% lower** than its current CTR. This is an observed historical outcome, not a claim about Google's ranking algorithm.

### Baseline
The baseline score prioritizes pages with lower current CTR, higher impression volume, and worse average position.

### Model
A Random Forest classifier predicts the probability of the defined future CTR-decline outcome.

### Validation
The model and baseline are evaluated on the **same held-out test set**. Entire clients are kept together using `GroupShuffleSplit`, so test clients are not mixed with training clients.

### Leakage check
All model features come from the prediction window. Future-window values are used only for the label. Client IDs are used only for grouping and are not model features. Product decision fields are not used.

In [ ]:
# Build the page-level modeling dataset.

current_window = con.sql(f'''
WITH current_window AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_current,
        SUM(gsc_clicks) AS clicks_current,
        AVG(gsc_avg_position) AS position_current,
        SUM(ga4_sessions) AS sessions_current,
        SUM(ga4_engaged_sessions) AS engaged_sessions_current,
        SUM(scroll_events) AS scroll_events_current,
        COUNT(DISTINCT report_date) AS days_current
    FROM read_parquet('{warehouse}')
    WHERE report_date BETWEEN '2026-03-01' AND '2026-04-30'
    GROUP BY client_hash_id, content_hash_id
),
future_window AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_future,
        SUM(gsc_clicks) AS clicks_future
    FROM read_parquet('{warehouse}')
    WHERE report_date BETWEEN '2026-05-01' AND '2026-06-30'
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    c.*,
    f.impressions_future,
    f.clicks_future,
    CASE
        WHEN c.impressions_current > 0
        THEN c.clicks_current * 1.0 / c.impressions_current
        ELSE NULL
    END AS ctr_current,
    CASE
        WHEN f.impressions_future > 0
        THEN f.clicks_future * 1.0 / f.impressions_future
        ELSE NULL
    END AS ctr_future
FROM current_window c
JOIN future_window f
  ON c.client_hash_id = f.client_hash_id
 AND c.content_hash_id = f.content_hash_id
WHERE c.impressions_current >= 100
  AND f.impressions_future >= 100
''').df()

current_window["ctr_change"] = (
    (current_window["ctr_future"] - current_window["ctr_current"])
    / current_window["ctr_current"].replace(0, np.nan)
)

current_window["target"] = (
    current_window["ctr_change"] <= -0.20
).astype(int)

print("Rows:", len(current_window))
print("CTR-decline rows:", current_window["target"].sum())
print("Target rate:", round(current_window["target"].mean(), 3))

display(current_window.head())

In [ ]:
# Grouped client-level holdout evaluation

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

model_df = current_window[
    current_window["ctr_current"].notna()
    & (current_window["ctr_current"] > 0)
].copy()

features = [
    "impressions_current",
    "clicks_current",
    "position_current",
    "sessions_current",
    "engaged_sessions_current",
    "scroll_events_current",
    "days_current",
    "ctr_current",
]

X = model_df[features].fillna(0)
y = model_df["target"]
groups = model_df["client_hash_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Training clients:", groups.iloc[train_idx].nunique())
print("Test clients:", groups.iloc[test_idx].nunique())

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

model.fit(X_train, y_train)
ml_probability = model.predict_proba(X_test)[:, 1]

# Simple baseline: lower CTR + higher visibility + worse position.
baseline_score = (
    (1 / (X_test["ctr_current"] + 1e-6))
    * np.log1p(X_test["impressions_current"])
    * (1 + X_test["position_current"] / 10)
)

def precision_at_50(scores, labels):
    order = np.argsort(scores)[::-1][:50]
    return labels.iloc[order].mean()

baseline_precision_at_50 = precision_at_50(
    baseline_score.to_numpy(),
    y_test.reset_index(drop=True)
)

model_precision_at_50 = precision_at_50(
    ml_probability,
    y_test.reset_index(drop=True)
)

print("Baseline Precision@50:", round(baseline_precision_at_50, 3))
print("ML Precision@50:", round(model_precision_at_50, 3))

## 4. Results (vs baseline)

The model and baseline are evaluated on the same client-held-out test set. Precision@50 measures the share of the top 50 ranked pages that match the defined observed future CTR-decline outcome.

In [ ]:
results = pd.DataFrame({
    "Method": ["Baseline", "ML model"],
    "Precision@50": [
        baseline_precision_at_50,
        model_precision_at_50
    ]
})

display(results)

## 5. Limitations

This analysis uses observed historical data and cannot prove that refreshing a page causes future performance to improve.

The rankings are directional decision-support, not causal predictions, and they do not predict Google's algorithm. Search performance can also be affected by factors that are not represented in the available fields.

A high-ranked page can therefore still be a poor refresh candidate and should be reviewed by a person before action.

In [ ]:
limitations = [
    "Observed historical data; not causal proof.",
    "Does not predict Google's algorithm.",
    "Some relevant factors may not be represented.",
    "Rankings are decision-support and require human review."
]

for item in limitations:
    print("-", item)

## 6. Ranked recommendations

For the action queue, a final Random Forest is trained on the available labeled modeling data using only prediction-window features. It produces a ranked review queue with a generic action and reason code.

The score is a prioritization signal, not a guarantee that a page should be changed.

In [ ]:
# Train a final model for the recommendation queue.

final_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

final_model.fit(X, y)

recommendations = model_df[
    ["client_hash_id", "content_hash_id"] + features
].copy()

recommendations["score"] = final_model.predict_proba(
    recommendations[features].fillna(0)
)[:, 1]

recommendations["action"] = np.where(
    recommendations["score"] >= 0.70,
    "Review / refresh",
    "Monitor"
)

recommendations["reason_code"] = np.where(
    recommendations["ctr_current"] < recommendations["ctr_current"].median(),
    "LOW_CURRENT_CTR",
    "DECLINE_RISK"
)

recommendations = recommendations.sort_values(
    "score",
    ascending=False
).reset_index(drop=True)

recommendations["rank"] = np.arange(
    1, len(recommendations) + 1
)

ranked_output = recommendations[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "score",
        "action",
        "reason_code"
    ]
].copy()

os.makedirs("work/outputs", exist_ok=True)

ranked_output.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

display(ranked_output.head(20))
print("Saved: work/outputs/baseline_action_score.csv")

## 7. Artifacts the paper embeds

The paper uses the model-versus-baseline chart, the validation table, and the ranked recommendation output generated above.

In [ ]:
import matplotlib.pyplot as plt

os.makedirs("work/outputs/charts", exist_ok=True)

ax = results.plot(
    x="Method",
    y="Precision@50",
    kind="bar",
    legend=False
)

plt.title("Model vs Baseline")
plt.ylabel("Precision@50")
plt.xlabel("")
plt.tight_layout()

chart_path = "work/outputs/charts/model_vs_baseline.png"
plt.savefig(chart_path, dpi=150, bbox_inches="tight")
plt.show()

print("Chart saved:", chart_path)

In [ ]:
print("=== CAPSTONE SANITY CHECK ===")
print("Rows evaluated:", len(X_test))
print("Baseline Precision@50:", round(baseline_precision_at_50, 3))
print("Model Precision@50:", round(model_precision_at_50, 3))
print("Top recommendations:", len(ranked_output.head(20)))

print("\nOutput files:")
print("- work/outputs/baseline_action_score.csv")
print("- work/outputs/charts/model_vs_baseline.png")

## ML-12 — 5-Minute Demo

1. **Problem:** Prioritize content pages for refresh review.
2. **Data:** Page-level search and engagement signals from the FlyRank internship warehouse.
3. **Method:** Compare a simple baseline with a Random Forest model using a client-held-out split.
4. **Results:** Compare Precision@50.
5. **Recommendations:** Show the ranked queue and reason codes.
6. **Limitations:** Explain that the results are directional decision-support, not causal proof.

## Social-Post Cut

I built a Search Intelligence workflow that ranks content pages for refresh review using observed search and engagement signals. I compared a simple baseline with an ML model and turned the results into a ranked decision-support queue. The project focuses on measurable, reproducible analysis rather than claims about Google's algorithm or causal impact.

## Employer-Facing Summary

I built an end-to-end Search Intelligence workflow for ranking content refresh opportunities. The project covers data framing, feature selection, a baseline, ML modeling, client-held-out validation, leakage checks, and ranked recommendations. The result is a reproducible decision-support workflow with honest limits on what the data can prove.

## Self-check

Before submitting:

- Every capstone section is filled.
- The notebook runs top-to-bottom without errors **after entering the Hugging Face read token interactively**.
- No token, client names, domains, URLs, private queries, or raw exports are included.
- Claims use careful language such as observed, measured, directional, and decision-support.
- `work/outputs/baseline_action_score.csv` and `work/outputs/charts/model_vs_baseline.png` are generated by the notebook.
- The notebook is committed under `work/notebooks/`.
- The deployed paper contains the required paper sections, including Abstract and Acknowledgments & data credit.
- `submission/paper_url.txt` contains the direct deployed-paper URL.
- ML-12 closing material is included.